Leave-one-source-out (LOSO) split generator

In [ ]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

full_df = pd.read_csv(DATA_DIR / "data_final.csv", low_memory=False)
full_df = full_df[full_df["category"] != "Undefined"].reset_index(drop=True)

with open(DATA_DIR / "artifacts" / "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
LABEL_ORDER = list(label_encoder.classes_)

SOURCES = ["df1", "df5"]
print("Sources:", SOURCES)
print("Rows after dropping Undefined:", len(full_df))

Sources: ['df1', 'df5']
Rows after dropping Undefined: 321184


In [2]:
VAL_FRACTION_WITHIN_TRAINVAL = 0.15  # 85/15 stratified split of the 5 non-held-out sources

fold_summary = []

for held_out in SOURCES:
    test_fold = full_df[full_df["source_dataset"] == held_out].reset_index(drop=True)
    trainval_fold = full_df[full_df["source_dataset"] != held_out].reset_index(drop=True)

    train_fold, val_fold = train_test_split(
        trainval_fold,
        test_size=VAL_FRACTION_WITHIN_TRAINVAL,
        stratify=trainval_fold["category"],
        random_state=42,
    )

    fold_dir = SPLITS_DIR / f"loso_{held_out}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    cols = ["text", "category", "source_dataset"]
    train_fold[cols].reset_index(drop=True).to_csv(fold_dir / "train.csv", index=False, encoding="utf-8")
    val_fold[cols].reset_index(drop=True).to_csv(fold_dir / "val.csv", index=False, encoding="utf-8")
    test_fold[cols].reset_index(drop=True).to_csv(fold_dir / "test.csv", index=False, encoding="utf-8")

    classes_present = sorted(test_fold["category"].unique())
    fold_summary.append(
        {
            "held_out_source": held_out,
            "n_train": len(train_fold),
            "n_val": len(val_fold),
            "n_test": len(test_fold),
            "n_classes_present": len(classes_present),
            "classes_present": classes_present,
        }
    )
    print(f"loso_{held_out}: train={len(train_fold)} val={len(val_fold)} test={len(test_fold)} classes={len(classes_present)}")

fold_summary_df = pd.DataFrame(fold_summary)
fold_summary_df

loso_df1: train=254600 val=44930 test=21654 classes=8


loso_df5: train=171082 val=30191 test=119911 classes=4


,held_out_source,n_train,n_val,n_test,n_classes_present,classes_present
0,df1,254600,44930,21654,8,"[Business, Entertainment, Health, Politics, Sc..."
1,df5,171082,30191,119911,4,"[Business, Sports, Technology, World]"


In [3]:
fold_summary_df.to_csv(SPLITS_DIR / "loso_fold_summary.csv", index=False, encoding="utf-8")
print("Wrote:", (SPLITS_DIR / "loso_fold_summary.csv").resolve())

Wrote: D:\UoL\Sem2\mnu2\models\data\splits\loso_fold_summary.csv
